In [ ]:
import os
from pathlib import Path
import joblib
import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px

# Parametros

In [ ]:
ambiente = 'dev'
costa = 'Matamoros'
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 2
RANDOM_SEED = 0
SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
MODEL_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s', 'wave_steepness']
SORT_FEATURES = ["wave_height_m", "wind_speed_ms", "wave_steepness"]

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_clasificator/wave_clasificator_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    model_path = f'{base_path}/wave_clasificator/wave_clasificator_{{}}.pkl'
    

# Obtener datos

In [ ]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{costa}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [ ]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}

data_sample = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

In [ ]:
fig = px.scatter_matrix(
    data_sample, 
    dimensions=MODEL_FEATURES
)
fig.update_layout(
    title=f'Correlación de variables de la costa {costa}',
    width=1200, 
    height=700
)
fig.show()

In [ ]:
correlation_matrix = data_sample[MODEL_FEATURES].corr()
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto")
fig.update_layout(
    title=f'Matriz de correlación de variables de la costa {costa}'
)
fig.show()

# Escalar

In [ ]:
scaler_path = scaler_path.format(costa)
scaler = joblib.load(scaler_path)

In [ ]:
scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

In [ ]:
scaled_df

# Gaussian Mixture

In [ ]:
from sklearn.mixture import GaussianMixture

## Parametros

In [ ]:
N_CLUSTERS = 6

In [ ]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [ ]:
data_sample["cluster"] = gmm.fit_predict(scaled_df[MODEL_FEATURES])
data_sample["cluster_probability"] = gmm.predict_proba(scaled_df[MODEL_FEATURES]).max(axis=1)

In [ ]:
data_sample

In [ ]:
cluster_summary = (
    data_sample.groupby("cluster")[MODEL_FEATURES]
    .mean()
    .sort_values(SORT_FEATURES)
)

cluster_order = {
    old_cluster: new_cluster + 1
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

data_sample["sea_state_level"] = data_sample["cluster"].map(cluster_order)

EXTREME_FEATURES = ['wind_speed_ms', 'wave_height_m']
data_sample['mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(0.99)
for feature in EXTREME_FEATURES[1:]:
    data_sample['mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(0.99)

data_sample.loc[data_sample['mask_extremo'], 'sea_state_level'] = 7

In [ ]:
data_sample['sea_state_level'].value_counts(normalize=True)*100

In [ ]:
data_sample

In [ ]:
sea_state_names = {
    1: "Mar calmado",
    2: "Mar suave",
    3: "Mar dinámico",
    4: "Mar agitado",
    5: "Mar fuerte",
    6: "Mar peligroso",
    7: "Mar extremo"
}

data_sample["sea_state"] = data_sample["sea_state_level"].map(sea_state_names)

In [ ]:
data_sample['sea_state'].value_counts(normalize=True)*100

In [ ]:
fig = px.scatter_3d(
    data_sample, 
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_height_m',
    color='sea_state'
)
fig.update_layout(
    title=f'Muestra datos de la costa {costa}',
    uirevision='constant',
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Altura de la ola (m)',
        aspectmode='cube'
    ),
    width=800, 
    height=700
)
fig.update_traces(marker=dict(size=3))
fig.show()

# Random Forest

In [ ]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)

In [ ]:
TARGET = "sea_state_level"

In [ ]:
X = data_sample[MODEL_FEATURES]
y = data_sample[TARGET].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

In [ ]:
y_pred = rf_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

In [ ]:
pd.DataFrame(rf_classifier.predict(X), columns=['predicted_sea_state_level']).value_counts(normalize=True)*100

In [ ]:
model_path = model_path.format(costa)
joblib.dump(rf_classifier, model_path)